# RudriQ: Tracing Why AI Systems Fail

**The problem.** When your production AI gives a wrong answer, the cause is usually upstream — in the data pipeline, not the model. Existing observability tools start at the LLM call. They can't tell you that last Tuesday's filter change is why today's answer is wrong.

**What RudriQ does.** Connects every data operation to every LLM call into one causal trace, scores the quality of each answer, ranks the likely root cause of failures, and produces a self-hosted, audit-grade report — running entirely inside your environment. No data leaves your VPC.

This notebook runs a realistic RAG pipeline end to end and shows exactly that — in one scroll.

## One line of setup

Your existing pandas + OpenAI code runs unchanged. The only RudriQ-specific line is `import rudriq.auto`. Everything else below is just a standard RAG pipeline.

In [ ]:
import contextlib
import io
import logging
import os
import re

os.environ['RUDRIQ_CAPTURE_CONTENT'] = 'true'
# Demo runs offline — no API keys needed. Suppress the upstream
# OpenLLMetry / OTLP exporter warnings about the 'demo' endpoint.
os.environ.setdefault('TRACELOOP_API_KEY', 'tl_demo_offline_no_real_key')
for _name in (
    'opentelemetry.exporter.otlp',
    'opentelemetry.exporter.otlp.proto.http.trace_exporter',
    'opentelemetry.exporter.otlp.proto.http.metric_exporter',
    'opentelemetry.sdk._logs._internal.export',
    'opentelemetry.sdk.metrics._internal.export',
    'opentelemetry.sdk.trace.export',
):
    logging.getLogger(_name).setLevel(logging.CRITICAL)

# Capture both stdout and stderr while the SDK pieces initialize, then
# emit our own clean status line. The third-party banners (Traceloop's
# "exporting traces to api.traceloop.com" line and OTLP exporter warnings)
# would otherwise crowd out the user-facing message AND imply a cloud
# dependency the tool doesn't actually have.
_stdout_buf, _stderr_buf = io.StringIO(), io.StringIO()
with contextlib.redirect_stdout(_stdout_buf), contextlib.redirect_stderr(_stderr_buf):
    import rudriq.auto  # one import — wires AutoLineage + OpenLLMetry + the linker + the span processor

    from opentelemetry import trace
    from rudriq.processors import RudriQSpanProcessor
    from rudriq.storage import get_default_storage

    provider = trace.get_tracer_provider()
    rudriq_proc = RudriQSpanProcessor()
    if hasattr(provider, 'add_span_processor'):
        provider.add_span_processor(rudriq_proc)

# Pull AutoLineage's "N hooks installed across M frameworks" line back
# out of the captured stdout — it's real social proof that the
# instrumentation actually wired itself in, distinct from "the import
# didn't raise."
_al_match = re.search(
    r'AutoLineage:\s*(\d+\s+hooks\s+installed\s+across\s+\d+\s+frameworks[^\n]*)',
    _stdout_buf.getvalue(),
)
_al_line = _al_match.group(1) if _al_match else 'hooks installed'

run_id = rudriq_proc.run_id
print(f'RudriQ initialized.')
print(f'  AutoLineage: {_al_line}')
print(f'  Span processor: registered with the global TracerProvider')
print(f'  Storage: ~/.rudriq/traces.duckdb (local, no outbound calls)')
print(f'  Run ID for this notebook: {run_id}')


## A realistic pipeline

5 source documents, filtered and joined through pandas, embedded in batch, and used as retrieval context for 10 chat completions. ~100 operations total, 11 LLM calls. **Standard code — no RudriQ-specific instrumentation, no decorators, no wrapped clients.** The cell below is the kind of pipeline a healthcare AI team writes today.

In [ ]:
import json
import time
from pathlib import Path
import tempfile
import httpx
import numpy as np
import pandas as pd
from openai import OpenAI

# 1. Mock the OpenAI HTTP layer so the notebook runs offline and is
#    reproducible. In a real deployment this is a real OpenAI client.
def _handler(req):
    body = json.loads(req.content) if req.content else {}
    if 'embeddings' in str(req.url):
        n = len(body.get('input', [])) or 1
        return httpx.Response(200, json={
            'object': 'list',
            'data': [{'object': 'embedding', 'index': i,
                      'embedding': [0.01 * (i + j) for j in range(8)]} for i in range(n)],
            'model': 'text-embedding-3-small',
            'usage': {'prompt_tokens': n * 5, 'total_tokens': n * 5},
        })
    return httpx.Response(200, json={
        'id': 'chatcmpl-mock', 'object': 'chat.completion', 'created': int(time.time()),
        'model': 'gpt-4',
        'choices': [{'index': 0, 'finish_reason': 'stop',
                     'message': {'role': 'assistant',
                                 'content': 'Based on the provided documents, the answer relates to the retrieved context.'}}],
        'usage': {'prompt_tokens': 100, 'completion_tokens': 20, 'total_tokens': 120},
    })
client = OpenAI(api_key='sk-test', http_client=httpx.Client(transport=httpx.MockTransport(_handler)))

# 2. Source documents (5 CSVs, 50 rows each — simulating clinical guideline shards).
tmp = Path(tempfile.gettempdir()) / 'rudriq_demo_clinical'
tmp.mkdir(exist_ok=True)
for i in range(5):
    p = tmp / f'guidelines_{i}.csv'
    if not p.exists():
        pd.DataFrame({
            'doc_id': [f'gl{i}_{j}' for j in range(50)],
            'text': [f'clinical guideline {i}.{j} on topic {j % 10}' for j in range(50)],
            'lang': ['en' if j % 3 != 0 else 'es' for j in range(50)],
            'score': np.random.RandomState(i).rand(50),
        }).to_csv(p, index=False)

# 3. Standard pandas transformations.
frames = []
for i in range(5):
    df = pd.read_csv(tmp / f'guidelines_{i}.csv')
    df = df[df['lang'] == 'en']
    df = df.drop_duplicates(subset=['text'])
    df = df.sort_values('score', ascending=False).head(15)
    frames.append(df)
combined = pd.concat(frames, ignore_index=True)
final_docs = combined.sort_values('score', ascending=False).reset_index(drop=True)
texts = final_docs['text'].tolist()

# 4. Batch-embed the corpus, then 10 retrieval-augmented chat completions.
# Suppress the offline-mock OTLP-exporter warnings so the cell output
# stays focused on what the user did, not the telemetry plumbing.
with contextlib.redirect_stderr(io.StringIO()):
    client.embeddings.create(model='text-embedding-3-small', input=texts)
    for topic in range(10):
        retrieved = '\n'.join(texts[topic*2:topic*2+3])
        client.chat.completions.create(
            model='gpt-4',
            messages=[
                {'role': 'system', 'content': 'Answer using only the provided guidelines.'},
                {'role': 'user', 'content': f'Context:\n{retrieved}\n\nQuestion: Tell me about topic {topic}'},
            ],
        )

print(f'Pipeline complete. {len(texts)} guideline snippets, 1 batch embedding, 10 chat completions.')

## The unified cross-domain trace

Standard LLM observability would show 11 LLM calls in isolation. RudriQ shows the unified graph — every data operation **and** every LLM call, linked. This is the seam no other tool covers.

In [ ]:
from rudriq.core.schema import EdgeKind, NodeKind

g = get_default_storage().load_run(run_id)
data_nodes = [n for n in g.nodes if n.kind.value.startswith('data_')]
llm_nodes = [n for n in g.nodes if n.kind.value.startswith('llm_')]
lineage_edges = [e for e in g.edges if e.kind == EdgeKind.LINEAGE_LINK]
linked_children = {e.child_id for e in lineage_edges}
linked = sum(1 for n in llm_nodes if n.node_id in linked_children)

print(f'  Total operations captured       : {len(g.nodes)}')
print(f'    Data operations (pandas)      : {len(data_nodes)}')
print(f'    LLM operations (openai)       : {len(llm_nodes)}')
print(f'  Cross-domain lineage links      : {len(lineage_edges)}')
print(f'  LLM calls linked to upstream    : {linked} / {len(llm_nodes)}')
print()
print('The batch embedding links to the data via object identity; chat completions link via')
print('substring match against tracked content. The honest part: query embeddings (if any) stay')
print('unlinked because they are freshly-constructed strings with no upstream tracked source —')
print('linking on coincidental similarity would produce false positives that destroy the trust')
print('value of the audit report.')

## Walk one answer back to its source

Pick a chat completion. Here's every operation that fed it — from the retrieval all the way back to the source CSV. The trace tells the story end to end.

In [ ]:
from rudriq.export.audit import _compute_lineage_chains  # internal helper, used here for the demo

chains = _compute_lineage_chains(g)
deepest = max(chains, key=lambda c: c['chain_length']) if chains else None
if deepest and deepest['chain']:
    print(f"LLM call : {deepest['llm_library']}.{deepest['llm_operation']}")
    print(f"Node     : {deepest['llm_node_id']}")
    print(f"Upstream depth: {deepest['chain_length']} operations")
    print()
    print('Upstream chain (most recent first):')
    for step in deepest['chain']:
        indent = '  ' * step['depth']
        print(f"{indent}- {step['library']}.{step['operation']}  ({step['kind']})")
else:
    print('(No deep upstream chains in this run — see Trace cell above for what was captured.)')

## Was each answer any good?

Five local evaluators score the run. **No cloud calls.** Embeddings run locally via fastembed (~130 MB ONNX, no torch). Each metric carries a status (green / yellow / red / gray) and a per-node explanation.

In [ ]:
from rudriq.export.audit import _run_audit_evaluations, _summarize_evaluations

eval_dicts = _run_audit_evaluations(g)
summary = _summarize_evaluations(eval_dicts)

lights = {'green': '[ ok  ]', 'yellow': '[warn ]', 'red': '[fail ]', 'gray': '[ na  ]'}
print(f"{'metric':22s} {'status':8s} {'mean':>6s} {'evaluated':>12s}")
print('-' * 52)
for metric in sorted(summary):
    s = summary[metric]
    light = lights.get(s['traffic_light'], '[ ?? ]')
    mean = s.get('mean_score')
    mean_str = f'{mean:.2f}' if mean is not None else '  N/A'
    applicable = s.get('applicable_total', s.get('total', 0))
    print(f"{metric:22s} {light}  {mean_str:>5s}  {s['evaluated']:>4d} / {applicable:<5d}")

## When something scores low, what caused it?

RudriQ auto-selects the worst LLM call (lowest groundedness) and ranks the likely upstream culprits — by structural deviation against a baseline (if provided), proximity in the lineage chain, and the confidence of the lineage links along the path.

**This is a ranked-suspects heuristic, not a causal proof.** The framing matters: for an audit-grade tool, claiming causation when you mean correlation is exactly the kind of overclaim that loses trust. We surface the evidence and let the human decide.

In [ ]:
from rudriq.analyzer.deviation_rca import DeviationRCA
from rudriq.export.audit import _auto_select_failure_target

# For the demo, target the LLM call with the deepest upstream chain (the
# batch embedding) — that's where RCA has the most ancestors to rank.
# By default the audit report's --include-rca auto-selects the lowest-
# groundedness chat node; here we pick the most diagnostic one to show
# what the ranking looks like with real upstream depth.
target = deepest['llm_node_id'] if (deepest and deepest['chain']) else _auto_select_failure_target(g)
print(f'Diagnosing: node {target}\n')

candidates = DeviationRCA().analyze(g, target)[:5]
if not candidates:
    print('No upstream operations linked to the target.')
else:
    print(f"{'#':>2}  {'operation':32s} {'score':>6s}  {'hops':>5s}  evidence")
    print('-' * 80)
    for i, c in enumerate(candidates, 1):
        op = f"{c.library}.{c.operation}"[:32]
        ev_keys = sorted(c.evidence.keys())
        ev = ', '.join(f'{k}={c.evidence[k]}' for k in ev_keys[:3])
        print(f"{i:>2}. {op:32s} {c.score:>6.3f}  {c.chain_distance:>5d}  {ev}")


## The deliverable — one audit report you hand a regulator

Combined: summary, lineage chains, quality evaluation, root-cause analysis, full operations appendix. Available as Markdown, JSON, or PDF. **Byte-deterministic** — hash the file to prove it's the exact artifact the system produced.

Below: produce all three forms; assert the PDF is byte-identical across two consecutive exports (the audit-grade property).

In [ ]:
import hashlib
from rudriq.export.audit import export_audit_json, export_audit_markdown
from rudriq.export.pdf import export_audit_pdf

md = export_audit_markdown(run_id, include_evals=True, include_rca=True)
js = export_audit_json(run_id, include_evals=True, include_rca=True)

out_dir = Path(tempfile.gettempdir()) / 'rudriq_demo_output'
out_dir.mkdir(exist_ok=True)
(out_dir / 'audit.md').write_text(md, encoding='utf-8')
(out_dir / 'audit.json').write_text(js, encoding='utf-8')
export_audit_pdf(run_id, str(out_dir / 'audit_a.pdf'), include_evals=True, include_rca=True)
export_audit_pdf(run_id, str(out_dir / 'audit_b.pdf'), include_evals=True, include_rca=True)

pdf_a = (out_dir / 'audit_a.pdf').read_bytes()
pdf_b = (out_dir / 'audit_b.pdf').read_bytes()
hash_a = hashlib.sha256(pdf_a).hexdigest()

print(f'Markdown report : {len(md):,} bytes  -> {out_dir / "audit.md"}')
print(f'JSON report     : {len(js):,} bytes  -> {out_dir / "audit.json"}')
print(f'PDF report      : {len(pdf_a):,} bytes  -> {out_dir / "audit_a.pdf"}')
print()
print(f'PDF sha256      : {hash_a}')
print(f'Reproducible    : {pdf_a == pdf_b} (two consecutive exports byte-identical)')

## What you just saw

- **One import.** Your code unchanged.
- **Cross-domain trace.** Data operations linked to LLM calls — the seam no other tool covers.
- **Local quality scoring.** Five evaluators, no cloud round-trip.
- **Honest root-cause ranking.** Ranked suspects with evidence, framed as a heuristic — not a causal claim.
- **A deterministic, hashable audit report** for EU AI Act, AI-liability underwriting, and litigation defense.
- **Entirely self-hosted.** No data left this environment. Air-gapped capable.

Built on [AutoLineage](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6683825) (published research). RudriQ is in active development; we're looking for design partners in regulated AI who'll shape where it goes.

**Contact:** [Kishan Raj VG](https://github.com/kishanraj41) · [github.com/kishanraj41/rudriq](https://github.com/kishanraj41/rudriq)